# 評価と図示 (Transformer vs LSTM)

In [ ]:
import os, sys, pickle
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))
from src.evaluate import compute_bleu, bleu_by_length

with open('../results/history_transformer.pkl', 'rb') as f:
    hist_t = pickle.load(f)
with open('../results/history_lstm.pkl', 'rb') as f:
    hist_l = pickle.load(f)
with open('../results/translations_test.pkl', 'rb') as f:
    trans = pickle.load(f)

print('Transformer params:', hist_t['n_params'])
print('LSTM params:       ', hist_l['n_params'])
print('Transformer total time:', hist_t['total_time'], 's')
print('LSTM total time:       ', hist_l['total_time'], 's')

## 学習曲線

In [ ]:
epochs = range(1, len(hist_t['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, hist_t['train_loss'], label='Transformer train')
axes[0].plot(epochs, hist_t['val_loss'],   label='Transformer val')
axes[0].plot(epochs, hist_l['train_loss'], label='LSTM train', linestyle='--')
axes[0].plot(epochs, hist_l['val_loss'],   label='LSTM val',   linestyle='--')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('cross-entropy loss')
axes[0].set_title('Learning curves'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, hist_t['val_loss'], label='Transformer val')
axes[1].plot(epochs, hist_l['val_loss'], label='LSTM val', linestyle='--')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('val loss')
axes[1].set_title('Validation loss only'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../results/loss_curve.png', dpi=120, bbox_inches='tight')
plt.show()

## BLEU

In [ ]:
bleu_t = compute_bleu(trans['transformer'], trans['refs'])
bleu_l = compute_bleu(trans['lstm'],        trans['refs'])
print(f'BLEU (Transformer): {bleu_t:.2f}')
print(f'BLEU (LSTM):        {bleu_l:.2f}')

## 文長別 BLEU

In [ ]:
buckets = ((1, 5), (6, 10), (11, 15), (16, 99))
by_len_t = bleu_by_length(trans['transformer'], trans['refs'], trans['src_lens'], buckets)
by_len_l = bleu_by_length(trans['lstm'],        trans['refs'], trans['src_lens'], buckets)

labels = [r['range'] for r in by_len_t]
scores_t = [r['bleu'] if r['bleu'] is not None else 0 for r in by_len_t]
scores_l = [r['bleu'] if r['bleu'] is not None else 0 for r in by_len_l]
ns       = [r['n']    for r in by_len_t]

import numpy as np
x = np.arange(len(labels)); w = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - w/2, scores_t, w, label='Transformer')
ax.bar(x + w/2, scores_l, w, label='LSTM')
ax.set_xticks(x); ax.set_xticklabels([f'{l}\n(n={n})' for l, n in zip(labels, ns)])
ax.set_xlabel('source length (tokens)')
ax.set_ylabel('BLEU')
ax.set_title('BLEU by source length')
ax.legend(); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/bleu_by_length.png', dpi=120, bbox_inches='tight')
plt.show()

for t, l in zip(by_len_t, by_len_l):
    if t['bleu'] is None: continue
    print(f"src_len {t['range']:>6} (n={t['n']}): Transformer {t['bleu']:.2f} | LSTM {l['bleu']:.2f}")

## 訳例の対比

In [ ]:
for i in [0, 1, 2, 10, 50, 100]:
    if i >= len(trans['refs']): continue
    print(f'--- sample {i} ---')
    print('REF :', trans['refs'][i])
    print('TRA :', trans['transformer'][i])
    print('LSTM:', trans['lstm'][i])
    print()

## まとめ表

In [ ]:
print('| モデル | パラメータ数 | 訓練時間 | 最終 train loss | 最終 val loss | BLEU |')
print('|---|---|---|---|---|---|')
print(f"| Transformer | {hist_t['n_params']:,} | {hist_t['total_time']:.0f}s | {hist_t['train_loss'][-1]:.3f} | {hist_t['val_loss'][-1]:.3f} | {bleu_t:.2f} |")
print(f"| LSTM        | {hist_l['n_params']:,} | {hist_l['total_time']:.0f}s | {hist_l['train_loss'][-1]:.3f} | {hist_l['val_loss'][-1]:.3f} | {bleu_l:.2f} |")